# Data Curation & Tokenizer Design

The transformer architecture is the same everywhere.
The attention mechanism is the same everywhere.
What makes one model better than another?

1. **Data** — what it learns from (the biggest factor)
2. **Tokenizer** — what units it sees (more important than it looks)

A perfect architecture trained on garbage data produces garbage.
A decent architecture trained on curated data produces something useful.

---
## Part 1: Why Data Curation Is #1

Training = learning patterns from data.
If the data contains false patterns, the model learns false patterns.

```
Training data says "the earth is flat" 1000 times?
  -> Model learns: earth is flat (confident!)

Training data says "the earth is round" 1000 times?
  -> Model learns: earth is round (confident!)
```

The model has no way to tell truth from lies.
It only knows what appears frequently in its training data.

In [ ]:
import math
import random

random.seed(42)

# Simulate: a model trained on different data distributions
# The model learns the frequency of patterns in its data

def train_on_data(data):
    """Simple frequency model: learns P(answer | question) from data."""
    counts = {}
    for question, answer in data:
        if question not in counts:
            counts[question] = {}
        counts[question][answer] = counts[question].get(answer, 0) + 1
    
    # Convert to probabilities
    probs = {}
    for q, answers in counts.items():
        total = sum(answers.values())
        probs[q] = {a: c / total for a, c in answers.items()}
    return probs

def predict(model, question):
    if question not in model:
        return "(unknown)", 0.0
    answers = model[question]
    best = max(answers, key=answers.get)
    return best, answers[best]

# Dataset A: uncurated internet scrape (contains misinformation)
uncurated_data = [
    ("capital of France", "Paris")] * 80 + [
    ("capital of France", "London")] * 15 + [
    ("capital of France", "Berlin")] * 5 + [
    ("earth shape", "round")] * 50 + [
    ("earth shape", "flat")] * 45 + [
    ("earth shape", "hollow")] * 5 + [
    ("vaccine safety", "safe and effective")] * 40 + [
    ("vaccine safety", "dangerous")] * 35 + [
    ("vaccine safety", "mind control")] * 25
]

# Dataset B: curated (filtered for quality sources)
curated_data = [
    ("capital of France", "Paris")] * 98 + [
    ("capital of France", "London")] * 2 + [
    ("earth shape", "round")] * 97 + [
    ("earth shape", "flat")] * 3 + [
    ("vaccine safety", "safe and effective")] * 95 + [
    ("vaccine safety", "dangerous")] * 5
]

model_uncurated = train_on_data(uncurated_data)
model_curated = train_on_data(curated_data)

print("Same architecture, different data:")
print("=" * 65)
print()

questions = ["capital of France", "earth shape", "vaccine safety"]

for q in questions:
    ans_u, conf_u = predict(model_uncurated, q)
    ans_c, conf_c = predict(model_curated, q)
    
    print(f"  Q: {q}")
    print(f"    Uncurated: '{ans_u}' ({conf_u:.0%} confident)")
    # Show full distribution
    for a, p in sorted(model_uncurated[q].items(), key=lambda x: -x[1]):
        bar = '#' * int(p * 30)
        print(f"      {a:<25} {p:>5.0%} {bar}")
    print(f"    Curated:   '{ans_c}' ({conf_c:.0%} confident)")
    for a, p in sorted(model_curated[q].items(), key=lambda x: -x[1]):
        bar = '#' * int(p * 30)
        print(f"      {a:<25} {p:>5.0%} {bar}")
    print()

print("  The uncurated model is uncertain about basic facts.")
print("  45% chance it says the earth is flat!")
print("  The curated model is confident and correct.")
print("  Same architecture. Only the data changed.")

---
## Part 2: What Data Curation Means in Practice

Raw internet data is messy:

```
- Duplicate pages (same article copied 100 times)
- Spam and SEO garbage ("best cheap flights click here")
- Misinformation (conspiracy theories, fake science)
- Toxic content (hate speech, harassment)
- Low quality (auto-generated, broken English)
- Personal data (emails, phone numbers, addresses)
- Outdated facts ("the president is..." from 2005)
```

Curation = filtering, weighting, and balancing this into useful training data.

In [ ]:
print("Data Curation Pipeline:")
print("=" * 65)
print()
print("  Raw Internet Crawl")
print("  |")
print("  | Step 1: DEDUPLICATION")
print("  | Remove exact and near-duplicate documents.")
print("  | Why: duplicates make the model memorize, not generalize.")
print("  | Method: MinHash / SimHash fingerprinting")
print("  | Impact: removes ~30-50% of raw data")
print("  |")
print("  | Step 2: QUALITY FILTERING")
print("  | Remove low-quality text.")
print("  | Signals: perplexity score, length, formatting,")
print("  |          ratio of punctuation, presence of boilerplate")
print("  | A classifier trained on 'good' text (Wikipedia, books)")
print("  | scores each document. Low score = remove.")
print("  | Impact: removes ~20-40% more")
print("  |")
print("  | Step 3: CONTENT FILTERING")
print("  | Remove harmful content.")
print("  | Toxic language classifier, NSFW filter,")
print("  | personal information detector (PII removal)")
print("  | Impact: removes ~5-10%")
print("  |")
print("  | Step 4: DOMAIN WEIGHTING")
print("  | Not all data is equal. Upweight valuable sources:")
print("  |   Wikipedia, textbooks, academic papers  -> weight x3")
print("  |   Code (GitHub, StackOverflow)            -> weight x2")
print("  |   Books                                   -> weight x2")
print("  |   News articles                           -> weight x1")
print("  |   Social media, forums                    -> weight x0.5")
print("  |   SEO spam, auto-generated                -> weight x0 (remove)")
print("  |")
print("  | Step 5: BALANCE")
print("  | Ensure diversity of topics, languages, styles.")
print("  | Too much code -> bad at conversation.")
print("  | Too much chat -> bad at reasoning.")
print("  |")
print("  v")
print("  Curated Training Dataset")
print()
print("  Typical ratio:")
print("  Raw crawl:  ~100 TB of text")
print("  After curation: ~5-15 TB of high-quality text")
print("  Only 5-15% survives. The rest is noise.")

In [ ]:
# Simulate the effect of each curation step

print("Effect of each curation step on model quality:")
print("=" * 65)
print()

# Simulate with a simple task: predict the correct answer
# Each curation step removes bad examples

raw_data = [
    # Good data
    ("2+2", "4")] * 40 + [
    ("capital of Japan", "Tokyo")] * 35 + [
    ("water formula", "H2O")] * 30 + [
    # Duplicates (same thing repeated)
    ("buy cheap shoes", "click here")] * 50 + [
    # Low quality
    ("2+2", "5")] * 10 + [
    ("2+2", "22")] * 8 + [
    # Misinformation
    ("capital of Japan", "Beijing")] * 12 + [
    ("water formula", "H3O")] * 5 + [
    # Toxic/harmful
    ("insult", "harmful content")] * 15
]

steps = [
    ("Raw (no curation)", raw_data),
]

# Step 1: Remove duplicates
after_dedup = [d for d in raw_data if d != ("buy cheap shoes", "click here")]
steps.append(("+ Deduplicate", after_dedup))

# Step 2: Quality filter (remove wrong answers)
bad_answers = {("2+2", "5"), ("2+2", "22")}
after_quality = [d for d in after_dedup if d not in bad_answers]
steps.append(("+ Quality filter", after_quality))

# Step 3: Content filter (remove misinformation)
misinfo = {("capital of Japan", "Beijing"), ("water formula", "H3O")}
after_content = [d for d in after_quality if d not in misinfo]
steps.append(("+ Content filter", after_content))

# Step 4: Remove toxic
after_toxic = [d for d in after_content if d[0] != "insult"]
steps.append(("+ Toxic filter", after_toxic))

test_questions = ["2+2", "capital of Japan", "water formula"]

print(f"  {'Step':<25} {'Size':>5}  ", end="")
for q in test_questions:
    print(f"  {q:<16}", end="")
print()
print(f"  {'-'*25} {'-'*5}  ", end="")
for _ in test_questions:
    print(f"  {'-'*16}", end="")
print()

for step_name, data in steps:
    model = train_on_data(data)
    print(f"  {step_name:<25} {len(data):>5}  ", end="")
    for q in test_questions:
        ans, conf = predict(model, q)
        ok = "ok" if ans in ["4", "Tokyo", "H2O"] else "XX"
        print(f"  {ans:<8} {conf:>4.0%} {ok} ", end="")
    print()

print()
print("  Raw data: 205 examples. After curation: 105 examples.")
print("  Half the data removed, but accuracy goes UP.")
print("  Less data, better model. Quality > quantity.")

---
## Part 3: Domain Weighting — Not All Data Is Equal

You don't just filter. You also **weight** data by importance.

If you want a model good at coding, show it more code.
If you want a model good at medicine, show it more medical text.
But don't over-specialize — the model also needs general knowledge.

In [ ]:
print("Data mix determines what the model is good at:")
print("=" * 65)
print()

# Three different data mixes
mixes = {
    "General (GPT-4)": {
        "Web text":     30,
        "Books":        20,
        "Code":         20,
        "Academic":     15,
        "Conversation": 10,
        "Math":          5,
    },
    "Code-focused (Codex)": {
        "Code":         60,
        "Documentation":15,
        "StackOverflow": 10,
        "Web text":     10,
        "Academic":      5,
    },
    "Medical (Med-PaLM)": {
        "Medical papers":  35,
        "Clinical notes":  25,
        "Medical textbooks":15,
        "General text":    15,
        "Drug databases":  10,
    },
}

for name, mix in mixes.items():
    print(f"  {name}:")
    for source, pct in mix.items():
        bar = '#' * (pct // 2)
        print(f"    {source:<20} {pct:>3}%  {bar}")
    print()

print("  Same transformer architecture in all three.")
print("  The data mix makes one a generalist, one a coder,")
print("  and one a doctor.")

In [ ]:
# Simulate: how data weighting affects model skills

# Each data point teaches the model one skill
# More examples = higher confidence in that skill

def simulate_training(mix, total_examples=1000):
    """Simulate model competence based on data mix."""
    skills = {}
    for source, pct in mix.items():
        n = int(total_examples * pct / 100)
        # Competence = log curve (diminishing returns)
        competence = min(1.0, 0.3 * math.log(n + 1))
        skills[source] = round(competence, 2)
    return skills

print("Resulting skill levels (0.0 = incompetent, 1.0 = expert):")
print("=" * 65)
print()

all_sources = set()
for mix in mixes.values():
    all_sources.update(mix.keys())

# Show skills for each model
for name, mix in mixes.items():
    skills = simulate_training(mix)
    print(f"  {name}:")
    for source, level in sorted(skills.items(), key=lambda x: -x[1]):
        bar = '#' * int(level * 30)
        print(f"    {source:<20} {level:.2f}  {bar}")
    print()

print("  The general model is OK at everything, great at nothing.")
print("  The code model excels at code but struggles with medicine.")
print("  The medical model excels at medicine but can't code well.")
print()
print("  This is why there are specialized models for each domain.")

---
## Part 4: The Tokenizer Is Not "Just Words to Numbers"

The tokenizer decides what **units** the model sees.
This fundamentally limits what it can learn.

A human analogy:

```
Bad tokenizer  = reading with broken glasses
                 You see fragments: "un" "hap" "pi" "ness"
                 Hard to understand the word.

Good tokenizer = reading with clear glasses
                 You see whole units: "unhappiness"
                 Instantly meaningful.
```

The model can only learn patterns between tokens it can see.
If the tokenizer breaks meaningful units, the model can't recover.

In [ ]:
# Show how different tokenizers see the same text

text = "The unhappy programmer's debugging session crashed unexpectedly."

print("Same sentence, three different tokenizers:")
print("=" * 65)
print(f'\n  "{text}"\n')

# Character-level tokenizer
char_tokens = list(text)
print(f"  1. Character-level ({len(char_tokens)} tokens):")
print(f"     {char_tokens[:20]}...")
print(f"     Problem: 'u','n','h','a','p','p','y' — 7 tokens for 1 word.")
print(f"     Model must learn to ASSEMBLE characters into meaning.")
print(f"     Very slow (long sequences). Very hard to learn.")
print()

# Word-level tokenizer
word_tokens = text.replace("'", " '").replace(".", " .").split()
print(f"  2. Word-level ({len(word_tokens)} tokens):")
print(f"     {word_tokens}")
print(f"     Problem: 'unexpectedly' is ONE token.")
print(f"     If the model never saw 'unexpectedly' in training,")
print(f"     it's a completely unknown word. No fallback.")
print(f"     Vocabulary would need millions of entries.")
print()

# Subword tokenizer (BPE-style)
subword_tokens = ["The", "un", "happy", "programm", "er", "'s",
                  "debug", "ging", "session", "crash", "ed",
                  "unexpected", "ly", "."]
print(f"  3. Subword/BPE ({len(subword_tokens)} tokens):")
print(f"     {subword_tokens}")
print(f"     'un' + 'happy' — model knows 'un' means negation.")
print(f"     'crash' + 'ed' — model knows '-ed' means past tense.")
print(f"     'unexpected' + 'ly' — model knows '-ly' means adverb.")
print(f"     Can handle NEW words by combining known pieces.")
print()
print("  Subword tokenization (BPE) is what GPT, BERT, Claude all use.")
print("  It balances vocabulary size with meaningful units.")

---
## Part 5: How BPE Tokenization Works

Byte-Pair Encoding (BPE) builds vocabulary from the bottom up:

1. Start with individual characters
2. Find the most frequent adjacent pair
3. Merge that pair into one new token
4. Repeat until vocabulary reaches target size

The result: common words become single tokens,
rare words get split into known subwords.

In [ ]:
# BPE from scratch

def get_pair_counts(tokenized_corpus):
    """Count adjacent token pairs across all words."""
    counts = {}
    for word_tokens in tokenized_corpus:
        for i in range(len(word_tokens) - 1):
            pair = (word_tokens[i], word_tokens[i+1])
            counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge_pair(tokenized_corpus, pair):
    """Merge all occurrences of pair into a single token."""
    merged_token = pair[0] + pair[1]
    new_corpus = []
    for word_tokens in tokenized_corpus:
        new_word = []
        i = 0
        while i < len(word_tokens):
            if (i < len(word_tokens) - 1 and
                word_tokens[i] == pair[0] and
                word_tokens[i+1] == pair[1]):
                new_word.append(merged_token)
                i += 2
            else:
                new_word.append(word_tokens[i])
                i += 1
        new_corpus.append(new_word)
    return new_corpus

# Training corpus
words = ["low", "low", "low", "low", "low",
         "lower", "lower",
         "newest", "newest", "newest", "newest",
         "widest", "widest", "widest"]

# Start: every word split into characters
corpus = [list(w) for w in words]
vocab = set()
for w in corpus:
    vocab.update(w)

print("BPE Tokenization: Building Vocabulary")
print("=" * 65)
print(f"\n  Training words: {list(set(words))}")
print(f"  Initial vocab: {sorted(vocab)}")
print(f"  Initial tokens: each character is a separate token")
print()

# Show unique words tokenized
unique_words = list(dict.fromkeys(words))
unique_corpus = [list(w) for w in unique_words]
print(f"  Starting state:")
for w, tokens in zip(unique_words, unique_corpus):
    print(f"    '{w}' -> {tokens}")
print()

# Run BPE merges
for step in range(8):
    pairs = get_pair_counts(corpus)
    if not pairs:
        break
    best_pair = max(pairs, key=pairs.get)
    count = pairs[best_pair]
    merged = best_pair[0] + best_pair[1]
    
    corpus = merge_pair(corpus, best_pair)
    unique_corpus = merge_pair(unique_corpus, best_pair)
    vocab.add(merged)
    
    print(f"  Merge {step+1}: '{best_pair[0]}' + '{best_pair[1]}' -> '{merged}' (appeared {count} times)")
    for w, tokens in zip(unique_words, unique_corpus):
        print(f"    '{w}' -> {tokens}")
    print()

print(f"  Final vocab ({len(vocab)} tokens): {sorted(vocab, key=len, reverse=True)[:10]}...")
print()
print("  Common words ('low') merged into single tokens first.")
print("  Shared suffixes ('est') also merged — the model can")
print("  now recognize '-est' means superlative in any word.")

---
## Part 6: Why Tokenizer Design Matters for Different Domains

A tokenizer trained on English text will butcher:
- Korean text (splits every syllable)
- Code (splits operators and keywords weirdly)
- Medical terms (splits "acetaminophen" into fragments)
- Math ("3.14159" becomes multiple tokens)

Domain-specific tokenizers are crucial.

In [ ]:
# Show how a general tokenizer handles different domains

# Simulated tokenization (based on typical BPE behavior)
domain_examples = [
    {
        "domain": "English (designed for this)",
        "text": "The cat sat on the mat.",
        "good_tokens": ["The", "cat", "sat", "on", "the", "mat", "."],
        "bad_tokens": ["The", "cat", "sat", "on", "the", "mat", "."],
    },
    {
        "domain": "Korean",
        "text": "고양이가 매트 위에 앉았다",
        "good_tokens": ["고양이가", "매트", "위에", "앉았다"],
        "bad_tokens": ["\xea", "\xb3", "\xa0", "\xec", "\x96", "\x91",
                       "\xec", "\x9d", "\xb4", "..."],
    },
    {
        "domain": "Python code",
        "text": "def fibonacci(n): return n if n<=1 else fibonacci(n-1)",
        "good_tokens": ["def", "fibonacci", "(", "n", "):", "return",
                        "n", "if", "n", "<=", "1", "else",
                        "fibonacci", "(", "n", "-", "1", ")"],
        "bad_tokens": ["def", "fib", "on", "acci", "(", "n", "):",
                       "return", "n", "if", "n", "<", "=", "1",
                       "else", "fib", "on", "acci", "(", "n", "-", "1", ")"],
    },
    {
        "domain": "Medical",
        "text": "Acetaminophen treats acute hepatotoxicity symptoms.",
        "good_tokens": ["Acetaminophen", "treats", "acute",
                        "hepato", "toxicity", "symptoms", "."],
        "bad_tokens": ["Ac", "et", "amin", "oph", "en", "treats",
                       "ac", "ute", "hep", "at", "ot", "ox",
                       "icity", "symptoms", "."],
    },
    {
        "domain": "Mathematics",
        "text": "3.14159 * r^2 = area",
        "good_tokens": ["3.14159", "*", "r", "^2", "=", "area"],
        "bad_tokens": ["3", ".", "14", "15", "9", "*",
                       "r", "^", "2", "=", "area"],
    },
]

print("General vs Domain-Specific Tokenizer:")
print("=" * 65)

for ex in domain_examples:
    print(f"\n  {ex['domain']}:")
    print(f"    Text: {ex['text']}")
    print(f"    General tokenizer ({len(ex['bad_tokens'])} tokens):")
    print(f"      {ex['bad_tokens']}")
    print(f"    Domain tokenizer  ({len(ex['good_tokens'])} tokens):")
    print(f"      {ex['good_tokens']}")

print("\n  The general tokenizer splits 'fibonacci' into 'fib','on','acci'.")
print("  The model must waste capacity learning to reassemble it.")
print("  A code-aware tokenizer keeps 'fibonacci' as one token.")
print("\n  Same problem for Korean, medical terms, and math.")
print("  The tokenizer shapes what the model can see and learn.")

In [ ]:
# Quantify the cost: bad tokenization = more tokens = slower + harder

print("The hidden cost of bad tokenization:")
print("=" * 65)
print()

cases = [
    ("English text",   7,  7, "1.0x"),
    ("Korean text",    4, 10, "2.5x"),
    ("Python code",   18, 23, "1.3x"),
    ("Medical text",   7, 15, "2.1x"),
    ("Math formula",   6, 11, "1.8x"),
]

print(f"  {'Domain':<16} {'Good tok':>9} {'Bad tok':>8} {'Overhead':>9}  Problem")
print(f"  {'-'*16} {'-'*9} {'-'*8} {'-'*9}  {'-'*20}")

problems = [
    "none",
    "every byte is a token",
    "'fibonacci' split into 3",
    "medical terms shattered",
    "'3.14159' is 5 tokens",
]

for (domain, good, bad, overhead), problem in zip(cases, problems):
    print(f"  {domain:<16} {good:>9} {bad:>8} {overhead:>9}  {problem}")

print()
print("  More tokens per sentence means:")
print("    - Slower inference (more steps to generate)")
print("    - Higher cost (APIs charge per token)")
print("    - Shorter effective context (window fills up faster)")
print("    - Harder to learn (must reconstruct meaning from fragments)")
print()
print("  This is why OpenAI, Google, etc. spend significant")
print("  effort designing tokenizers for each language and domain.")

---
## Part 7: Non-Text Domains — Tokenization Is Everything

For images, audio, and protein, there's no natural "word".
The entire input representation IS the tokenization.

```
Text:    obvious units (words exist)
Image:   how to split? patches? pixels? objects?
Audio:   how to split? frames? phonemes? syllables?
Protein: how to split? single amino acids? motifs? domains?
```

The "tokenizer" for these domains IS the embedding model.
Get it wrong and the transformer sees meaningless fragments.

In [ ]:
print("Tokenization across domains:")
print("=" * 65)
print()

domains = [
    {
        "name": "Text",
        "raw": "'The cat sat'",
        "bad": "each character -> 11 tokens (T,h,e, ,c,a,t,...)",
        "good": "subword BPE -> 3 tokens (The, cat, sat)",
        "why": "words are natural meaning units",
    },
    {
        "name": "Image",
        "raw": "224x224 RGB image",
        "bad": "each pixel -> 150,528 tokens (way too many)",
        "good": "16x16 patches -> 196 tokens (manageable)",
        "why": "patches capture local patterns (edges, textures)",
    },
    {
        "name": "Audio",
        "raw": "3 sec speech at 16kHz",
        "bad": "each sample -> 48,000 tokens (raw waveform)",
        "good": "mel spectrogram frames -> 75 tokens",
        "why": "mel spec matches how humans hear frequencies",
    },
    {
        "name": "Protein",
        "raw": "amino acid sequence MKFLVTL...",
        "bad": "each atom -> thousands of tokens",
        "good": "each amino acid -> 1 token (20 types)",
        "why": "amino acids are the functional units",
    },
    {
        "name": "Music",
        "raw": "audio waveform",
        "bad": "raw samples -> millions of tokens",
        "good": "quantized codes (Encodec) -> ~75 tokens/sec",
        "why": "learned compression preserves musical structure",
    },
]

for d in domains:
    print(f"  {d['name']}:")
    print(f"    Raw input:          {d['raw']}")
    print(f"    Bad tokenization:   {d['bad']}")
    print(f"    Good tokenization:  {d['good']}")
    print(f"    Why it works:       {d['why']}")
    print()

print("  In every domain, the tokenization decision determines")
print("  whether the model sees meaningful units or noise.")
print("  This is why 'just pixels' doesn't work for vision")
print("  and 'just samples' doesn't work for audio.")

---
## Part 8: The Full Picture — What Makes a Good Model

Ranked by impact on model quality:

In [ ]:
print("What determines model quality (ranked):")
print("=" * 65)
print()

factors = [
    (1, "Data quality (curation)",
     "What it learns from. Garbage in = garbage out.",
     "Filter, deduplicate, weight, balance.",
     35),
    (2, "Tokenizer design",
     "What units it sees. Determines what patterns are learnable.",
     "Domain-appropriate subword units.",
     20),
    (3, "Training objective",
     "What it's asked to learn. Next-token vs masked vs contrastive.",
     "Must match your use case.",
     15),
    (4, "Data quantity (scale)",
     "How many examples. More data = more patterns learned.",
     "Diminishing returns after a point.",
     15),
    (5, "Model size (params)",
     "How much capacity. More params = finer distinctions.",
     "Must match data quantity.",
     10),
    (6, "Architecture details",
     "Dim, layers, heads, context window.",
     "Important but well-understood. Less room for innovation.",
     5),
]

for rank, name, desc, detail, impact in factors:
    bar = '#' * (impact // 1)
    print(f"  #{rank} {name} ({impact}%)")
    print(f"     {desc}")
    print(f"     {detail}")
    print(f"     Impact: {bar}")
    print()

print("  Data curation + tokenizer = 55% of model quality.")
print("  Architecture = 5%.")
print()
print("  This is why companies invest heavily in data pipelines")
print("  and tokenizer research, not in inventing new attention.")
print("  The attention mechanism hasn't changed since 2017.")
print("  Everything else around it has.")

---
## Summary

| Factor | Why it matters | Key insight |
|--------|---------------|-------------|
| **Data curation** | Model learns ONLY from training data | Quality > quantity. Filter ruthlessly. |
| **Deduplication** | Duplicates cause memorization, not learning | 30-50% of raw data is duplicates |
| **Quality filtering** | Low-quality text teaches low-quality patterns | Use classifier trained on good text |
| **Domain weighting** | More weight = better at that domain | Choose your mix to match your goal |
| **Tokenizer** | Determines what units the model sees | Bad splits = model sees fragments |
| **BPE** | Learns subword units from data | Common words = 1 token, rare = pieces |
| **Domain tokenizer** | Generic tokenizer butchers specialized text | Code, Korean, medical need custom tokenizers |

The transformer architecture is the engine.
Data curation is the fuel quality.
The tokenizer is the fuel injector.

A perfect engine with dirty fuel and a clogged injector goes nowhere.